In [1]:
import feedparser
import pandas as pd
import tmdbsimple as tmdb
from sklearn.metrics.pairwise import cosine_similarity
import time
import os
from dotenv import load_dotenv

tmdb.API_KEY = os.getenv("TMDB_API_KEY")
tmdb.REQUESTS_TIMEOUT = 5 

In [2]:
watched_df = pd.read_csv(r"E:\Python\movie-reccomender\ratings.csv")
 
# Parse date and clean up
watched_df["entry_published"] = pd.to_datetime(watched_df["Date"]).dt.strftime("%a, %-d %b %Y %H:%M:%S +0000")
watched_df = watched_df.rename(columns={"Name": "entry_title", "Rating": "entry_rating"})

In [3]:
def search_tmdb(title, year=None):
    """
    Try a movie search first, then fall back to TV.
    Returns (movie_id, tv_id) — one will always be NaN.
    """
    search = tmdb.Search()
 
    # Movie search
    kwargs = {"query": title}
    if year:
        kwargs["year"] = year
    search.movie(**kwargs)
    if search.results:
        return float(search.results[0]["id"]), float("nan")
 
    # TV search (no year filter — TMDB TV search ignores it anyway)
    search.tv(query=title)
    if search.results:
        return float("nan"), float(search.results[0]["id"])
 
    return float("nan"), float("nan")
 
 
movie_ids, tv_ids = [], []
 
for _, row in watched_df.iterrows():
    title = row["entry_title"]
    # Extract year from the Letterboxd "Year" column if present
    year = row.get("Year")
    year = int(year) if pd.notna(year) and str(year).isdigit() else None
 
    m_id, t_id = search_tmdb(title, year)
    movie_ids.append(m_id)
    tv_ids.append(t_id)
 
    time.sleep(0.25)   # stay well within TMDB rate limits (40 req/10 s)
 
watched_df["movie_id"] = movie_ids
watched_df["tv_id"]    = tv_ids

In [4]:
watched_df = watched_df[["entry_title", "entry_published", "entry_rating", "movie_id", "tv_id"]].copy()
watched_df = watched_df.sort_values("entry_published", ascending=False).reset_index(drop=True)
 

watched_df

,entry_title,entry_published,entry_rating,movie_id,tv_id
0,Checkpoint Zoo,2026-04-15 00:00:00,4.0,1176733.0,NaN
1,Project Hail Mary,2026-04-13 00:00:00,4.0,687163.0,NaN
2,The Drama,2026-04-06 00:00:00,4.0,1325734.0,NaN
3,Big Trouble in Little China,2026-02-14 00:00:00,3.0,6978.0,NaN
4,Scream,2026-02-14 00:00:00,4.0,4232.0,NaN
...,...,...,...,...,...
118,The Dark Knight,2024-02-15 00:00:00,5.0,155.0,NaN
119,Nightcrawler,2024-02-07 00:00:00,4.5,242582.0,NaN
120,Risky Business,2024-01-22 00:00:00,4.0,9346.0,NaN
121,The Wolf of Wall Street,2024-01-16 00:00:00,4.0,106646.0,NaN


In [5]:
movie_df = watched_df.dropna(subset = ["movie_id"])
movie_df['tv_id'] = pd.to_numeric(movie_df['tv_id'])
movie_df['movie_id'] = pd.to_numeric(movie_df['movie_id'])



movie_df['genres'] = None  # resets the column to object dtype

for movieId in movie_df['movie_id']:
    movie = tmdb.Movies(int(movieId))
    response = movie.info()
    idx = movie_df[movie_df['movie_id'] == movieId].index[0]
    movie_df.at[idx, 'genres'] = ', '.join([g['name'] for g in movie.genres])
    time.sleep(0.1)

movie_df

,entry_title,entry_published,entry_rating,movie_id,tv_id,genres
0,Checkpoint Zoo,2026-04-15 00:00:00,4.0,1176733.0,NaN,Documentary
1,Project Hail Mary,2026-04-13 00:00:00,4.0,687163.0,NaN,"Science Fiction, Adventure"
2,The Drama,2026-04-06 00:00:00,4.0,1325734.0,NaN,"Romance, Comedy, Drama"
3,Big Trouble in Little China,2026-02-14 00:00:00,3.0,6978.0,NaN,"Action, Adventure, Comedy, Fantasy"
4,Scream,2026-02-14 00:00:00,4.0,4232.0,NaN,"Crime, Horror, Mystery"
...,...,...,...,...,...,...
118,The Dark Knight,2024-02-15 00:00:00,5.0,155.0,NaN,"Action, Crime, Thriller"
119,Nightcrawler,2024-02-07 00:00:00,4.5,242582.0,NaN,"Crime, Drama, Thriller"
120,Risky Business,2024-01-22 00:00:00,4.0,9346.0,NaN,"Romance, Comedy, Drama"
121,The Wolf of Wall Street,2024-01-16 00:00:00,4.0,106646.0,NaN,"Crime, Drama, Comedy"


In [6]:
tv_df = watched_df.dropna(subset = ["tv_id"])
tv_df['movie_id'] = pd.to_numeric(tv_df['movie_id'])
tv_df['tv_id'] = pd.to_numeric(tv_df['tv_id'])

tv_df['genres'] = None  # resets the column to object dtype

for tvId in tv_df['tv_id']:
    tv = tmdb.TV(int(tvId))
    response = tv.info()
    idx = tv_df[tv_df['tv_id'] == tvId].index[0]
    tv_df.at[idx, 'genres'] = ', '.join([g['name'] for g in tv.genres])
    time.sleep(0.1)
tv_df

,entry_title,entry_published,entry_rating,movie_id,tv_id,genres
13,Frieren: Beyond Journey's End,2026-01-19 00:00:00,5.0,NaN,209867.0,"Animation, Action & Adventure, Drama, Sci-Fi &..."
60,Chainsaw Man,2025-05-19 00:00:00,4.0,NaN,114410.0,"Animation, Action & Adventure, Sci-Fi & Fantas..."


In [7]:
movie = tmdb.Movies()
movies = []
page = 1
seen_ids = set(watched_df['movie_id'].dropna().astype(int).tolist())

while len(movies) < 1000:
    response = movie.top_rated(page=page)
    for item in response['results']:
        if item['id'] not in seen_ids:
            movies.append(item)
    page += 1
    time.sleep(.1)

print(response["results"][0]["title"])

len(movies)

popular_df = pd.DataFrame(movies)
popular_df

Steamboat Bill, Jr.


,adult,backdrop_path,genre_ids,id,title,original_language,original_title,overview,popularity,poster_path,release_date,softcore,video,vote_average,vote_count
0,False,/zMwhWailP1WY7sb6AoE6b8ugoy.jpg,"[16, 10751, 12, 18, 14]",1007757,Swapped,en,Swapped,"A small woodland creature and a majestic bird,...",425.2006,/tHhxWxge06goXU6ZQH1hj7vK8Hd.jpg,2026-05-01,False,False,8.990,796
1,False,/zfbjgQE1uSd9wiPTX4VzsLi0rGG.jpg,"[18, 80]",278,The Shawshank Redemption,en,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,52.0334,/9cqNxx0GxF0bflZmeSMuL5tnGzr.jpg,1994-09-23,False,False,8.720,30319
2,False,/tSPT36ZKlP2WVHJLM4cQPLSzv3b.jpg,"[18, 80]",238,The Godfather,en,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",39.3318,/3bhkrj58Vtu7enYsRolD1fZdja1.jpg,1972-03-14,False,False,8.686,22880
3,False,/qO55CD8tgVL1T4WKn6zYFFiD6lL.jpg,"[28, 18, 80]",1439930,A Marvel Television Special Presentation - The...,en,A Marvel Television Special Presentation - The...,As Frank Castle searches for meaning beyond re...,306.4791,/gOggsBCSypNXq0yApYeXe7nnopT.jpg,2026-05-12,False,False,8.604,347
4,False,/kGzFbGhp99zva6oZODW5atUtnqi.jpg,"[18, 80]",240,The Godfather Part II,en,The Godfather Part II,In the continuing saga of the Corleone crime f...,26.7893,/hek3koDUyRQk7FIhPXsa6mT2Zc3.jpg,1974-12-20,False,False,8.571,13872
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1007,False,/lYfHa1AtkUMtplrKx7SRLHpwonW.jpg,"[12, 14, 27]",244,King Kong,en,King Kong,Adventurous filmmaker Carl Denham sets out to ...,4.0028,/lHlnxKL5GbgRibyRFI7n1Ey850i.jpg,1933-04-07,False,False,7.600,1608
1008,False,/9n2tJBplPbgR2ca05hS5CKXwP2c.jpg,"[10751, 35, 12, 14, 16]",502356,The Super Mario Bros. Movie,en,The Super Mario Bros. Movie,"While working underground to fix a water main,...",45.6474,/qNBAXBIQlnOThrVvA6mA2B5ggV6.jpg,2023-04-05,False,False,7.594,10637
1009,False,/xXhta1NIKn09IXy0mfp68cabdWS.jpg,"[35, 10749]",466282,To All the Boys I've Loved Before,en,To All the Boys I've Loved Before,Lara Jean's love life goes from imaginary to o...,5.3995,/hKHZhUbIyUAjcSrqJThFGYIR6kI.jpg,2018-08-17,False,False,7.594,8704
1010,False,/v8AmfO3BW4NT4SiNnsocKMzexOR.jpg,"[18, 35, 10749]",61202,Zindagi Na Milegi Dobara,hi,ज़िन्दगी ना मिलेगी दोबारा,Three friends who were inseparable in childhoo...,1.8577,/hKO9O715wYxjkQSEv47giCYcyO8.jpg,2011-07-15,False,False,7.594,405


In [8]:
#Import TfIdfVectorizer from scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer

#Define a TF-IDF Vectorizer Object. Remove all english stop words such as 'the', 'a'
tfidf = TfidfVectorizer(stop_words='english')

#Replace NaN with an empty string
popular_df['overview'] = popular_df['overview'].fillna('')

#Construct the required TF-IDF matrix by fitting and transforming the data
tfidf_matrix = tfidf.fit_transform(popular_df['overview'])

#Output the shape of tfidf_matrix
tfidf_matrix.shape

(1012, 8311)

In [9]:
# Import linear_kernel
from sklearn.metrics.pairwise import linear_kernel

# Compute the cosine similarity matrix
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

In [10]:
#Construct a reverse map of indices and movie titles
indices = pd.Series(popular_df.index, index=popular_df['title']).drop_duplicates()
popular_df

,adult,backdrop_path,genre_ids,id,title,original_language,original_title,overview,popularity,poster_path,release_date,softcore,video,vote_average,vote_count
0,False,/zMwhWailP1WY7sb6AoE6b8ugoy.jpg,"[16, 10751, 12, 18, 14]",1007757,Swapped,en,Swapped,"A small woodland creature and a majestic bird,...",425.2006,/tHhxWxge06goXU6ZQH1hj7vK8Hd.jpg,2026-05-01,False,False,8.990,796
1,False,/zfbjgQE1uSd9wiPTX4VzsLi0rGG.jpg,"[18, 80]",278,The Shawshank Redemption,en,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,52.0334,/9cqNxx0GxF0bflZmeSMuL5tnGzr.jpg,1994-09-23,False,False,8.720,30319
2,False,/tSPT36ZKlP2WVHJLM4cQPLSzv3b.jpg,"[18, 80]",238,The Godfather,en,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",39.3318,/3bhkrj58Vtu7enYsRolD1fZdja1.jpg,1972-03-14,False,False,8.686,22880
3,False,/qO55CD8tgVL1T4WKn6zYFFiD6lL.jpg,"[28, 18, 80]",1439930,A Marvel Television Special Presentation - The...,en,A Marvel Television Special Presentation - The...,As Frank Castle searches for meaning beyond re...,306.4791,/gOggsBCSypNXq0yApYeXe7nnopT.jpg,2026-05-12,False,False,8.604,347
4,False,/kGzFbGhp99zva6oZODW5atUtnqi.jpg,"[18, 80]",240,The Godfather Part II,en,The Godfather Part II,In the continuing saga of the Corleone crime f...,26.7893,/hek3koDUyRQk7FIhPXsa6mT2Zc3.jpg,1974-12-20,False,False,8.571,13872
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1007,False,/lYfHa1AtkUMtplrKx7SRLHpwonW.jpg,"[12, 14, 27]",244,King Kong,en,King Kong,Adventurous filmmaker Carl Denham sets out to ...,4.0028,/lHlnxKL5GbgRibyRFI7n1Ey850i.jpg,1933-04-07,False,False,7.600,1608
1008,False,/9n2tJBplPbgR2ca05hS5CKXwP2c.jpg,"[10751, 35, 12, 14, 16]",502356,The Super Mario Bros. Movie,en,The Super Mario Bros. Movie,"While working underground to fix a water main,...",45.6474,/qNBAXBIQlnOThrVvA6mA2B5ggV6.jpg,2023-04-05,False,False,7.594,10637
1009,False,/xXhta1NIKn09IXy0mfp68cabdWS.jpg,"[35, 10749]",466282,To All the Boys I've Loved Before,en,To All the Boys I've Loved Before,Lara Jean's love life goes from imaginary to o...,5.3995,/hKHZhUbIyUAjcSrqJThFGYIR6kI.jpg,2018-08-17,False,False,7.594,8704
1010,False,/v8AmfO3BW4NT4SiNnsocKMzexOR.jpg,"[18, 35, 10749]",61202,Zindagi Na Milegi Dobara,hi,ज़िन्दगी ना मिलेगी दोबारा,Three friends who were inseparable in childhoo...,1.8577,/hKO9O715wYxjkQSEv47giCYcyO8.jpg,2011-07-15,False,False,7.594,405


In [11]:
# Function that takes in movie title as input and outputs most similar movies
def get_recommendations(title, cosine_sim=cosine_sim):
    # Get the index of the movie that matches the title
    idx = indices[title]

    # Get the pairwsie similarity scores of all movies with that movie
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort the movies based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the 10 most similar movies
    sim_scores = sim_scores[1:11]

    # Get the movie indices
    movie_indices = [i[0] for i in sim_scores]

    # Return the top 10 most similar movies
    return popular_df['title'].iloc[movie_indices]

In [12]:
get_recommendations('The Avengers')

948           Kingsman: The Secret Service
923                             Zootopia 2
159    Lock, Stock and Two Smoking Barrels
158                           Paris, Texas
468                       A Beautiful Mind
744                   John Wick: Chapter 4
868                         The Bad Guys 2
933                           Mediterraneo
591            A Woman Under the Influence
488                         Thirteen Lives
Name: title, dtype: str

In [ ]:
# from concurrent.futures import ThreadPoolExecutor
# import threading

# from concurrent.futures import ThreadPoolExecutor

# def get_credits(movie_id):
#     try:
#         movie = tmdb.Movies(movie_id)
#         credits = movie.credits()
        
#         director = next(
#             (member["name"] for member in credits["crew"] if member["job"] == "Director"),
#             "Director not found"
#         )
#         actors = [member["name"] for member in credits["cast"][:5]]
#         keyworddict = movie.keywords()
#         keywords = [kw["name"] for kw in keyworddict["keywords"][:3]]
        
#         return director, actors, keywords
#     except Exception as e:
#         return "Error", []

# with ThreadPoolExecutor(max_workers=15) as executor:
#     results = list(executor.map(get_credits, popular_df["id"]))

# popular_df["director"], popular_df["actors"], popular_df["keywords"] = zip(*results)
# # popular_df

In [60]:
def clean_data(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    else:
        #Check if director exists. If not, return empty string
        if isinstance(x, str):
            return str.lower(x.replace(" ", ""))
        else:
            return ''



In [134]:
features = ['actors', 'keywords', 'director', 'genres', "original_title"]

genres = tmdb.Genres()
response = genres.movie_list()

merged = {d['id']: d['name'] for d in response['genres']}

popular_df['genres'] = popular_df['genre_ids'].apply(lambda ids: [merged[i] for i in ids if i in merged])

for feature in features:
    popular_df[feature] = popular_df[feature].apply(clean_data)

In [156]:
def create_soup(x):
    director = x['director'] + ' ' + x['director']
    genres = ' '.join(x['genres']) + ' ' + ' '.join(x['genres'])
    actors = ' '.join(x['actors'][:3])
    keywords = ' '.join(x['keywords'])
    return f"{director} {genres} {actors} {keywords}"
popular_df['soup'] = popular_df.apply(create_soup, axis=1)

idx = popular_df[popular_df['id'] == 497].index[0]
idx
popular_df.loc[idx, 'soup']

'frankdarabont frankdarabont fantasy drama crime fantasy drama crime tomhanks davidmorse bonniehunt mentallydisabled deathpenalty basedonnovelorbook'

In [68]:
from sklearn.feature_extraction.text import CountVectorizer

count = CountVectorizer(stop_words='english')
count_matrix = count.fit_transform(popular_df['soup'])

In [69]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim2 = cosine_similarity(count_matrix, count_matrix)

In [70]:
popular_df = popular_df.reset_index()
indices = pd.Series(popular_df.index, index=popular_df['title'])

In [80]:
get_recommendations('Evangelion: 3.0+1.0 Thrice Upon a Time', cosine_sim2)

567      Evangelion: 2.0 You Can (Not) Advance
717                    Cowboy Bebop: The Movie
940       Dragon Ball Z: The History of Trunks
354    Batman: The Dark Knight Returns, Part 2
691    Batman: The Dark Knight Returns, Part 1
439                                   La Jetée
559                                      Finch
810            Kizumonogatari Part 1: Tekketsu
514                      No Game No Life: Zero
535                          World of Tomorrow
Name: title, dtype: str

In [85]:
def get_recommendations_weighted(watched_titles_with_ratings, cosine_sim=cosine_sim2, top_n=10):
    """
    watched_titles_with_ratings: list of (title, rating) tuples
    e.g. [("The Avengers", 4.5), ("Parasite", 5.0)]
    """
    
    valid = [(indices[t], r) for t, r in watched_titles_with_ratings if t in indices]

    notmissing = [t for t, r in watched_titles_with_ratings if t in indices]
    if notmissing:
        print(f"{notmissing}")
    if not valid:
        return []

    watched_idx = {idx for idx, _ in valid}
    
    # Build a weighted sum of similarity rows
    total_weight = sum(r for _, r in valid)
    weighted_scores = sum(cosine_sim[idx] * (r / total_weight) for idx, r in valid)

    sim_scores = sorted(enumerate(weighted_scores), key=lambda x: x[1], reverse=True)
    sim_scores = [(i, s) for i, s in sim_scores if i not in watched_idx][:top_n]

    return popular_df['title'].iloc[[i for i, _ in sim_scores]]

In [86]:
watched_pairs = list(zip(watched_df['entry_title'], watched_df['entry_rating']))
get_recommendations_weighted(watched_pairs)

['Whiplash', 'Nosferatu']


388                        Faust
705                      Kwaidan
884                 Frankenstein
411                         CODA
635              Wings of Desire
162                         Soul
198                            M
616         The Phantom Carriage
428           My Father's Violin
63     Out of the Clear Blue Sky
Name: title, dtype: str

In [97]:
for movieId in movie_df['movie_id']:
    movie = tmdb.Movies(int(movieId))
    response = movie.info()
    idx = movie_df[movie_df['movie_id'] == movieId].index[0]
    movie_df.at[idx, 'overview'] = response['overview']
    time.sleep(0.1)
movie_df

,entry_title,entry_published,entry_rating,movie_id,tv_id,genres,overview,"(overview, False)"
0,Checkpoint Zoo,2026-04-15 00:00:00,4.0,1176733.0,NaN,Documentary,Checkpoint Zoo documents a daring rescue led b...,"In his second year of fighting crime, Batman u..."
1,Project Hail Mary,2026-04-13 00:00:00,4.0,687163.0,NaN,"Science Fiction, Adventure",Science teacher Ryland Grace wakes up on a spa...,"In his second year of fighting crime, Batman u..."
2,The Drama,2026-04-06 00:00:00,4.0,1325734.0,NaN,"Romance, Comedy, Drama",A happily engaged couple is put to the test wh...,"In his second year of fighting crime, Batman u..."
3,Big Trouble in Little China,2026-02-14 00:00:00,3.0,6978.0,NaN,"Action, Adventure, Comedy, Fantasy",Truck driver Jack Burton gets embroiled in a s...,"In his second year of fighting crime, Batman u..."
4,Scream,2026-02-14 00:00:00,4.0,4232.0,NaN,"Crime, Horror, Mystery","A year after the murder of her mother, a teena...","In his second year of fighting crime, Batman u..."
...,...,...,...,...,...,...,...,...
118,The Dark Knight,2024-02-15 00:00:00,5.0,155.0,NaN,"Action, Crime, Thriller",Batman raises the stakes in his war on crime. ...,"In his second year of fighting crime, Batman u..."
119,Nightcrawler,2024-02-07 00:00:00,4.5,242582.0,NaN,"Crime, Drama, Thriller","When Lou Bloom, desperate for work, muscles in...","In his second year of fighting crime, Batman u..."
120,Risky Business,2024-01-22 00:00:00,4.0,9346.0,NaN,"Romance, Comedy, Drama","Meet Joel Goodson, an industrious, college-bou...","In his second year of fighting crime, Batman u..."
121,The Wolf of Wall Street,2024-01-16 00:00:00,4.0,106646.0,NaN,"Crime, Drama, Comedy",A New York stockbroker refuses to cooperate in...,"In his second year of fighting crime, Batman u..."


In [99]:
from concurrent.futures import ThreadPoolExecutor
import threading

from concurrent.futures import ThreadPoolExecutor

def get_credits(movie_id):
    try:
        movie = tmdb.Movies(movie_id)
        credits = movie.credits()
        
        director = next(
            (member["name"] for member in credits["crew"] if member["job"] == "Director"),
            "Director not found"
        )
        actors = [member["name"] for member in credits["cast"][:5]]
        keyworddict = movie.keywords()
        keywords = [kw["name"] for kw in keyworddict["keywords"][:3]]
        
        return director, actors, keywords
    except Exception as e:
        return "Error", []

with ThreadPoolExecutor(max_workers=15) as executor:
    results = list(executor.map(get_credits, movie_df['movie_id']))

movie_df["director"], movie_df["actors"], movie_df["keywords"] = zip(*results)

In [104]:
movie_df

,entry_title,entry_published,entry_rating,movie_id,tv_id,genres,overview,"(overview, False)",director,actors,keywords
0,Checkpoint Zoo,2026-04-15 00:00:00,4.0,1176733.0,NaN,Documentary,Checkpoint Zoo documents a daring rescue led b...,"In his second year of fighting crime, Batman u...",Joshua Zeman,"[Oleksandr Feldman, Vadim Vorontinsky, Vitalii...","[zoo, animal rescue, political documentary]"
1,Project Hail Mary,2026-04-13 00:00:00,4.0,687163.0,NaN,"Science Fiction, Adventure",Science teacher Ryland Grace wakes up on a spa...,"In his second year of fighting crime, Batman u...",Phil Lord,"[Ryan Gosling, Sandra Hüller, James Ortiz, Lio...","[friendship, coma, based on novel or book]"
2,The Drama,2026-04-06 00:00:00,4.0,1325734.0,NaN,"Romance, Comedy, Drama",A happily engaged couple is put to the test wh...,"In his second year of fighting crime, Batman u...",Kristoffer Borgli,"[Zendaya, Robert Pattinson, Mamoudou Athie, Al...","[infidelity, boston, massachusetts, dark comedy]"
3,Big Trouble in Little China,2026-02-14 00:00:00,3.0,6978.0,NaN,"Action, Adventure, Comedy, Fantasy",Truck driver Jack Burton gets embroiled in a s...,"In his second year of fighting crime, Batman u...",John Carpenter,"[Kurt Russell, Kim Cattrall, Dennis Dun, James...","[martial arts, kung fu, magic]"
4,Scream,2026-02-14 00:00:00,4.0,4232.0,NaN,"Crime, Horror, Mystery","A year after the murder of her mother, a teena...","In his second year of fighting crime, Batman u...",Wes Craven,"[David Arquette, Neve Campbell, Courteney Cox,...","[high school, small town, riddle]"
...,...,...,...,...,...,...,...,...,...,...,...
118,The Dark Knight,2024-02-15 00:00:00,5.0,155.0,NaN,"Action, Crime, Thriller",Batman raises the stakes in his war on crime. ...,"In his second year of fighting crime, Batman u...",Christopher Nolan,"[Christian Bale, Heath Ledger, Aaron Eckhart, ...","[sadism, chaos, secret identity]"
119,Nightcrawler,2024-02-07 00:00:00,4.5,242582.0,NaN,"Crime, Drama, Thriller","When Lou Bloom, desperate for work, muscles in...","In his second year of fighting crime, Batman u...",Dan Gilroy,"[Jake Gyllenhaal, Riz Ahmed, Rene Russo, Bill ...","[underground, psychopath, journalism]"
120,Risky Business,2024-01-22 00:00:00,4.0,9346.0,NaN,"Romance, Comedy, Drama","Meet Joel Goodson, an industrious, college-bou...","In his second year of fighting crime, Batman u...",Paul Brickman,"[Tom Cruise, Rebecca De Mornay, Joe Pantoliano...","[chicago, illinois, high school, brothel]"
121,The Wolf of Wall Street,2024-01-16 00:00:00,4.0,106646.0,NaN,"Crime, Drama, Comedy",A New York stockbroker refuses to cooperate in...,"In his second year of fighting crime, Batman u...",Martin Scorsese,"[Leonardo DiCaprio, Jonah Hill, Margot Robbie,...","[corruption, based on novel or book, drug addi..."


In [110]:
features = ['actors', 'keywords', 'director', 'genres', 'entry_title']

for feature in features:
    movie_df[feature] = movie_df[feature].apply(clean_data)

In [155]:
def create_soup(x):
    director = x['director'] + ' ' + x['director']
    genres = ' '.join(x['genres']) + ' ' + ' '.join(x['genres'])
    actors = ' '.join(x['actors'][:3])
    keywords = ' '.join(x['keywords'])
    return f"{director} {genres} {actors} {keywords}"
movie_df['soup'] = movie_df.apply(create_soup, axis=1)


In [157]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Combine both dataframes for vectorization
all_soup = pd.concat([popular_df['soup'], movie_df['soup']], ignore_index=True)

# 2. Fit and transform on the combined set
count = CountVectorizer(stop_words='english')
count_matrix = count.fit_transform(all_soup)

# 3. Split the matrix back apart
n_popular = len(popular_df)
popular_matrix = count_matrix[:n_popular]   # candidate pool
watched_matrix = count_matrix[n_popular:]   # your watched movies

# 4. Build a weighted taste profile vector from your watched movies
ratings = movie_df['entry_rating']
weights = ratings / ratings.sum()
taste_profile = weights @ watched_matrix.toarray()  # single weighted vector

# 5. Score every candidate against your taste profile
scores = cosine_similarity([taste_profile], popular_matrix)[0]

# 6. Exclude movies you've already seen
watched_ids = set(movie_df['movie_id'].dropna().astype(int))
results = []
for idx, score in sorted(enumerate(scores), key=lambda x: x[1], reverse=True):
    if popular_df.iloc[idx]['id'] not in watched_ids:
        results.append(popular_df.iloc[idx]['title'])
    if len(results) == 10:
        break

results

['Oppenheimer',
 'Casino',
 'Shutter Island',
 "One Flew Over the Cuckoo's Nest",
 'The Martian',
 'Little Women',
 'The King of Comedy',
 'Raging Bull',
 'The Prestige',
 'The Hustler']

In [153]:
import numpy as np

nonzero = taste_profile[taste_profile > 0]
print(f"Total dimensions: {len(taste_profile)}")
print(f"Non-zero dimensions: {len(nonzero)}")
print(f"Max value: {nonzero.max():.4f}")
print(f"Top 10 values: {sorted(nonzero, reverse=True)[:10]}")
top_dim = np.argmax(taste_profile)
top_word = count.get_feature_names_out()[top_dim]
print(f"Top word: {top_word}")

Total dimensions: 6105
Non-zero dimensions: 896
Max value: 0.5060
Top 10 values: [np.float64(0.5060000000000002), np.float64(0.3510000000000002), np.float64(0.2820000000000001), np.float64(0.26800000000000007), np.float64(0.24200000000000008), np.float64(0.2250000000000001), np.float64(0.15700000000000003), np.float64(0.15500000000000003), np.float64(0.138), np.float64(0.13199999999999998)]
Top word: drama
